In [2]:
# --- A) Setup & Scaling for SVMs ---

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# 1) Detect column types from your existing X_train
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

print(f"Numeric cols: {len(num_cols)} | Categorical cols: {len(cat_cols)}")

# 2) Preprocess:
#    - Numeric: median impute + standardize (SVMs are scale-sensitive)
#    - Categorical (if any): most-frequent impute + one-hot encode (ignore unseen)
pre_svm = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop",
    n_jobs=-1
)

# 3) (Optional) Quick check: fit the preprocessor and transform a small batch
_ = pre_svm.fit_transform(X_train)  # fits on training only (no leakage)
print("Preprocessor ready for SVMs.")

NameError: name 'X_train' is not defined

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

svm_lin = Pipeline([
    ("pre", pre),  # your ColumnTransformer with StandardScaler
    ("clf", LinearSVC(C=1.0, loss="squared_hinge", random_state=42))
])
svm_lin.fit(X_train, y_train)

NameError: name 'pre' is not defined

In [ ]:
from sklearn.svm import SVC

svm_rbf = Pipeline([
    ("pre", pre),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=42))
])
svm_rbf.fit(X_train, y_train)

In [ ]:
svm_poly = Pipeline([
    ("pre", pre),
    ("clf", SVC(kernel="poly", degree=2, coef0=1.0, C=1.0, gamma="scale", probability=False, random_state=42))
])
svm_poly.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
from scipy.special import expit
import numpy as np

def get_scores(model, X):
    # prefer decision_function (SVMs); fallback to predict_proba if available
    if hasattr(model, "decision_function"):
        return model.decision_function(X)  # raw margins
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    # final fallback: use predict as score (not ideal for AUC)
    return model.predict(X)

def evaluate_clf(model, X_test, y_test, name="Model", threshold=0.5):
    y_score = get_scores(model, X_test)
    # convert score to probs if needed for thresholding visuals; for reporting, threshold on score directly is fine
    if y_score.ndim == 1:
        y_pred = (y_score >= 0).astype(int)  # SVM margin sign as default
    else:
        y_pred = (y_score >= threshold).astype(int)
    acc = accuracy_score(y_test, y_pred)
    # AUC expects real-valued scores: use y_score directly
    auc = roc_auc_score(y_test, y_score)
    print(f"{name} — Accuracy: {acc:.4f} | ROC AUC: {auc:.4f}\n")
    print("Classification report:\n", classification_report(y_test, y_pred))
    return acc, auc

In [ ]:
evaluate_clf(svm_lin,  X_test, y_test, name="Linear SVM")
evaluate_clf(svm_rbf,  X_test, y_test, name="RBF SVM")
evaluate_clf(svm_poly, X_test, y_test, name="Poly SVM (deg=2)")

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

param_grid = {
    "clf__C": [0.5, 1, 2],
    "clf__gamma": ["scale", 0.1, 0.01]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_rbf_gs = GridSearchCV(
    estimator=Pipeline([("pre", pre), ("clf", SVC(kernel="rbf", probability=False, random_state=42))]),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)
svm_rbf_gs.fit(X_train, y_train)
print("Best params:", svm_rbf_gs.best_params_, "CV AUC:", svm_rbf_gs.best_score_)
evaluate_clf(svm_rbf_gs.best_estimator_, X_test, y_test, name="RBF SVM (Best CV)")